# A proof-backed IMAS regridding check

The Lean theorem `RegridCheck.check_sound` says that acceptance by the executable checker implies that every stored target temperature is within the absolute tolerance of the piecewise-linear source interpolant on `rho_tor_norm`. The two selected profiles must have the same recorded time. This demo creates synthetic DD 4.1.1 profiles and writes them as IMAS records.

Use the repository's `.venv` Python kernel with Lean 4.35.0 available. The setup cell locates the repository when the notebook runs from either the repository root or its `notebooks/` folder. The tested Python dependencies can be installed in the notebook kernel with

```python
%pip install "imas-python==2.3.0" "imas-validator @ git+https://github.com/iterorganization/IMAS-Validator.git@52164a5832547c31bef613d379833cab700fb6ee"
```

This revision of IMAS-Validator works with IMAS-Python 2.3.0. The PyPI release 1.0.0 calls an API that IMAS-Python 2.3.0 removed.

In [1]:
import subprocess
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

build = subprocess.run(
    ["lake", "build", "lean4imas-regrid"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
if build.returncode:
    raise RuntimeError(build.stdout + build.stderr)
print("Lean checker built.")


Lean checker built.


## Create the profiles with IMAS-Python

In [2]:
import imas

factory = imas.IDSFactory("4.1.1")

def profile(radii, temperatures):
    ids = factory.core_profiles()
    ids.ids_properties.homogeneous_time = 1
    ids.time = [1.25]
    ids.profiles_1d.resize(1)
    point = ids.profiles_1d[0]
    point.grid.rho_tor_norm = radii
    point.electrons.temperature = temperatures
    return ids

source = profile([0.0, 0.25, 1.0], [1000.0, 1500.0, 2250.0])
target_radii = [0.0, 0.125, 0.25, 0.5, 1.0]
candidates = {
    "interpolated": profile(target_radii, [1000.0, 1250.0, 1500.0, 1750.0, 2250.0]),
    "changed final sample": profile(target_radii, [1000.0, 1250.0, 1500.0, 1750.0, 2250.5]),
}

16:43:46 INFO     Parsing data dictionary version 4.1.1 @dd_zip.py:89


## Check two candidate outputs

The source has three samples and two intervals with different slopes. Both candidate tests claim that the target is within `0.1 eV`. The second deliberately changes the final temperature by `0.5 eV`, so the checker should reject that proposed verdict.

### The proposition to audit

**RegridsWithinTolerance** states what a passing verdict means. Its definition and the supporting definitions are together in [RegridSpec.lean](../Lean4IMAS/Checks/RegridSpec.lean).

It requires the same finite time for both slices, exact correspondence with the stored radii and temperatures, a nonnegative tolerance, at least two strictly increasing source radii, and at least one target sample. Every target sample must lie between adjacent source radii and agree with their linear interpolant within the tolerance.

In Lean, a definition ending in `: Prop` describes a mathematical statement. Read `∧` as “and”, `∃` as “there exists”, and `∀ point ∈ targetSamples` as “for every stored target sample”.

```lean
def RegridsWithinTolerance (source target : Slice) (tolerance : Rat) : Prop :=
  ∃ time sourceSamples targetSamples,
    sliceTime source = some time ∧ sliceTime target = some time ∧
    SamplesOf source sourceSamples ∧ SamplesOf target targetSamples ∧
    0 ≤ tolerance ∧ 2 ≤ sourceSamples.length ∧ targetSamples ≠ [] ∧
    sourceSamples.Pairwise (fun left right => left.radius < right.radius) ∧
    ∀ point ∈ targetSamples, MatchesInterpolant sourceSamples tolerance point
```

`SamplesOf` ties the samples to the data. `MatchesInterpolant` specifies the bracketing and error bound, using the formula in `linearTemperature`. These definitions belong to the claim being audited. [Read the specification](../Lean4IMAS/Checks/RegridSpec.lean)

The implementation and the proof that acceptance implies this claim are in [RegridCheck.lean](../Lean4IMAS/Checks/RegridCheck.lean).

In [3]:
from lean4imas import check_regrid

tolerance_ev = 0.1
proposed_verdicts = {"interpolated": True, "changed final sample": True}

verdicts = {}
for name, target in candidates.items():
    verdict = check_regrid(source, target, tolerance_ev=tolerance_ev)
    verdicts[name] = verdict.passed
    label_agrees = verdict.passed == proposed_verdicts[name]
    print(f"{name}: proposed outcome agrees = {label_agrees}")
    print(f"  {'PASS' if verdict.passed else 'FAIL'} — {verdict.message}")

assert verdicts == {"interpolated": True, "changed final sample": False}

interpolated: proposed outcome agrees = True
  PASS — The stored target temperatures agree with the linear interpolant.
changed final sample: proposed outcome agrees = False
  FAIL — Target sample 4 is outside the source grid or exceeds the temperature tolerance.


### Inside `check_regrid`

1. **Python sends the data.** `lean4imas.py` selects a `profiles_1d` slice from each IDS and reads its time, `rho_tor_norm` grid, and electron temperatures. It sends these fields and the tolerance to the compiled `lean4imas-regrid` executable as JSON. The numbers are passed as their original binary64 bits to preserve the stored values.
2. **Lean decodes the slices.** `RegridMain.lean` fills the corresponding fields in two generated Lean records. `ExactFloat.lean` interprets finite floats as exact fractions, so, e.g. a stored `0.25` becomes exactly $1/4$.
3. **Lean checks the samples.** `RegridCheck.lean` checks that the times match and finds the adjacent source radii that bracket each target radius. It compares the stored target temperature with the linear interpolant. Python receives the verdict as `passed` and `message`.

`RegridSpec.lean` contains the mathematical statement about the decoded records. `check_sound` proves that whenever `check` succeeds, the statement holds for every stored target sample. The build checks that proof. Each run evaluates the executable check on its own data.

## Save the IDSs with IMAS-Python

The source goes into occurrence 0. Each candidate goes into occurrence 1 in its own HDF5 data entry. The notebook keeps the temporary directory available for the validation step. [IMAS-Python HDF5 documentation](https://imas-python.readthedocs.io/en/stable/intro.html#load-and-store-an-ids-to-disk-with-imas-core)

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory

demo_directory = TemporaryDirectory()
directory = Path(demo_directory.name)

source_uri = f"imas:hdf5?path={directory}/source"
with imas.DBEntry(source_uri, "w", dd_version="4.1.1") as entry:
    entry.put(source, occurrence=0)

target_uris = {}
for index, (name, target) in enumerate(candidates.items()):
    uri = f"imas:hdf5?path={directory}/target-{index}"
    with imas.DBEntry(uri, "w", dd_version="4.1.1") as entry:
        entry.put(target, occurrence=1)
    target_uris[name] = uri

## Define the IMAS-Validator rule

IMAS-Validator loads custom rules from Python files. The cell below saves the displayed rule in the demo's temporary directory. The runner supplies `@validator` when it loads that file.

In [5]:
rulesets_dir = directory / "rulesets"
rule_path = rulesets_dir / "lean4imas" / "core_profiles.py"
rule_path.parent.mkdir(parents=True, exist_ok=True)

In [6]:
%%writefile "{rule_path}"
"""Check core_profiles:1 against core_profiles:0 at LEAN4IMAS_SOURCE_URI."""

import os

import imas

from lean4imas import check_regrid


@validator("core_profiles:1")
def validate_temperature_regrid(target):
    """Check slice 0 of the source and target against a linear regrid to 0.1 eV."""
    source_uri = os.environ.get("LEAN4IMAS_SOURCE_URI")
    if not source_uri:
        raise ValueError("Set LEAN4IMAS_SOURCE_URI to the entry containing core_profiles:0.")
    # Load the source explicitly because the result collector requires distinct
    # IDS names for multiple arguments. The runner supplies the wrapped target.
    with imas.DBEntry(source_uri, "r", dd_version="4.1.1") as entry:
        source = entry.get("core_profiles", occurrence=0, autoconvert=False)
        result = check_regrid(source, target._obj, tolerance_ev=0.1)
    assert result.passed, result.message


## Run the same check through IMAS-Validator

The optional rule receives `core_profiles:1` from the entry under validation. It loads `core_profiles:0` from the entry named by `LEAN4IMAS_SOURCE_URI` and checks slice 0 from each. The rule calls the same `check_regrid` function used above. IMAS-Validator records `assert result.passed, result.message` as a rule result.

In [7]:
import os
from unittest.mock import patch

from imas_validator.validate.validate import validate
from imas_validator.validate_options import ValidateOptions

options = ValidateOptions(
    rulesets=["lean4imas"],
    extra_rule_dirs=[rulesets_dir],
    apply_generic=False,
    stop_at_load_error=True,
)

with patch.dict(os.environ, {"LEAN4IMAS_SOURCE_URI": source_uri}):
    for name, uri in target_uris.items():
        [result] = validate(uri, options).results
        assert result.exc is None
        assert result.success == verdicts[name]
        print(f"{name}: {result.rule.name} → {'PASS' if result.success else 'FAIL'}")

interpolated: lean4imas/core_profiles.py:validate_temperature_regrid → PASS
changed final sample: lean4imas/core_profiles.py:validate_temperature_regrid → FAIL
